In [12]:
import math
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import HTML

# 1. SETUP & CONSTANTS 
%matplotlib inline

mass_of_sun = 1.989e30 
mass_of_earth = 5.972e24 
G = 6.6743e-11 

# Orbital Parameters
earth_aphelion = 1.521e11 
earth_perihelion = 1.471e11 
eccentricity_earth = 0.0167
semi_major_axis = (earth_perihelion + earth_aphelion) / 2
semi_minor_axis = semi_major_axis * math.sqrt(1 - (eccentricity_earth**2))

# Asteroid Parameters (1986 DA)
AU = 1.496e11 
e_ast = 0.5848
a_ast = 2.8137 * AU       
q_ast = 1.1683 * AU       

# User Breifing

print ("Welcome to Asteroid Mining Simulation version 3.1") 
print ("Target Asteroid: 6178 (1986 DA)") 
print ("Requesting Primary Parameters...")
user_input = input("Enter Username:")
print ("Initializing")
print("-------------------------------")

# Time Settings 
dt = 60*60*24 # 1 Day
while True:
    try:
        year_input = input(f"{user_input}, enter your desiered simulation duration in years (10-80): ")
        user_years = float(year_input)
        
        if 10 <= user_years <= 80:
            break 
        else:
            print("Out of range! Please enter a number between 10 and 80.")
            
    except ValueError:
        print("Invalid format! Please enter a number (e.g., 40).")

total_days = int(user_years * 365)
num_steps = total_days

# Accuracy Settings
print ("Simulation Accuracy is scaled exponentially, please enter in accordance with simulation duration Ex:")
print ("If your simulation lasts 40 years, scale your simulation accuracy to around 60%")
print ("If your simulation lasts 50 years, scale your simulation accuracy to around 40%")
print ("**************** WARNING ****************")
print (" If you set your simulation accuracy too high with a large simulation duration, your system may crash")
print ("*****************************************")
while True:
    try:
        graphic_input = input(f"{user_input}, enter your desiered simulation accuracy in percent accuracy (10-90): ")
        graphics_quality_float = float(graphic_input)
        
        if 10 <= graphics_quality_float <= 90:
            break 
        else:
            print("Out of range! Please enter a number between 10 and 90.")
            
    except ValueError:
        print("Invalid format! Please enter a number (e.g., 5).")

graphics_quality = int(100 - graphics_quality_float)


# 2. INITIALIZE PLANETS 
# Earth
x_earth = earth_perihelion
y_earth = 0
vx = 0
vy = math.sqrt(G * mass_of_sun * (2/earth_perihelion - 1/semi_major_axis))

# Asteroid
x_ast = q_ast
y_ast = 0
vx_ast = 0
vy_ast = math.sqrt(G * mass_of_sun * (2/x_ast - 1/a_ast))

# Arrays to store the "Map" of the solar system
earth_pos = np.zeros((num_steps, 2))
ast_pos = np.zeros((num_steps, 2))

# Lists for plotting later
x_history, y_history = [], []
x_ast_history, y_ast_history = [], []


# 3. PLANETARY PHYSICS LOOP 
print("Initializing Orbit Calculation")
for i in range(num_steps):
    # Earth Physics
    r_earth = math.sqrt(x_earth**2 + y_earth**2)
    F_earth = G * mass_of_sun * mass_of_earth / r_earth**2
    ax_earth = -(F_earth * (x_earth / r_earth)) / mass_of_earth
    ay_earth = -(F_earth * (y_earth / r_earth)) / mass_of_earth
    
    vx += ax_earth * dt
    vy += ay_earth * dt
    x_earth += vx * dt
    y_earth += vy * dt
    
    # Asteroid Physics
    r_ast = math.sqrt(x_ast**2 + y_ast**2)
    acc_ast = (G * mass_of_sun) / r_ast**2
    ax_ast = -acc_ast * (x_ast / r_ast)
    ay_ast = -acc_ast * (y_ast / r_ast)
    
    vx_ast += ax_ast * dt
    vy_ast += ay_ast * dt
    x_ast += vx_ast * dt
    y_ast += vy_ast * dt
    
    # SAVE DATA TO ARRAYS (Crucial Step!)
    earth_pos[i] = [x_earth, y_earth]
    ast_pos[i] = [x_ast, y_ast]
    
    # Save to history lists for animation
    x_history.append(x_earth)
    y_history.append(y_earth)
    x_ast_history.append(x_ast)
    y_ast_history.append(y_ast)

print("Complete")
print("-------------------------------")
# 4. CALCULATE LAUNCH WINDOW 
print(" -Mission Control Startup- ") 
# We look for the closest approach
distances = np.linalg.norm(earth_pos - ast_pos, axis=1)

#Ignore simulation startup
future_distances = distances.copy()
future_distances[:500] = np.inf 

closest_index = np.argmin(future_distances)

# Launch window search parameters
test_start = closest_index - 100
test_end = closest_index + 100
best_day = 0
closest_miss_distance = float('inf')

print(f"Closest approach: Day {closest_index}")


print(f"Scanning launch windows from Day {test_start} to {test_end}...")
for test_day in range(test_start, test_end):
    # Skip if out of bounds
    if test_day < 0 or test_day >= num_steps: continue

    # Start Rocket State at Earth on this test day
    rx, ry = earth_pos[test_day]
    
    # Calculate Launch Vector (Earth Direction + Kick)
    # Get Earth Velocity
    evx = (earth_pos[test_day][0] - earth_pos[test_day-1][0]) / dt
    evy = (earth_pos[test_day][1] - earth_pos[test_day-1][1]) / dt
    e_speed = math.sqrt(evx**2 + evy**2)
    
    # Add Kick
    kick_velocity = 8000
    vx = evx + (evx/e_speed) * kick_velocity
    vy = evy + (evy/e_speed) * kick_velocity
    
    # Fast-Forward Simulation to find closest approach
    # We only run for 3 years (approx 1000 days) after launch to save time
    local_min_dist = float('inf')
    
    for t in range(test_day, min(test_day + 1500, num_steps)):
        # Update Rocket Physics (Sun Gravity Only)
        r = math.sqrt(rx**2 + ry**2)
        acc = -(G * mass_of_sun) / r**3
        vx += acc * rx * dt
        vy += acc * ry * dt
        rx += vx * dt
        ry += vy * dt
        
        # Check distance to asteroid at this time t
        ax, ay = ast_pos[t]
        dist = math.sqrt((rx - ax)**2 + (ry - ay)**2)
        
        if dist < local_min_dist:
            local_min_dist = dist
            
    # Record if this was the best day so far
    if local_min_dist < closest_miss_distance:
        closest_miss_distance = local_min_dist
        best_day = test_day

print(f"OPTIMAL LAUNCH FOUND: Day {best_day}")
print(f"Projected miss distance: {closest_miss_distance/1000:.0f} km")
print("-------------------------------")
# 5. RUN ROCKET SIMULATION 
x_rocket_hist = []
y_rocket_hist = []

rocket_x, rocket_y = 0, 0
rocket_vx, rocket_vy = 0, 0
state = "ON_EARTH"

vx_rocket_hist = []
vy_rocket_hist = []
current_vx = 0
current_vy = 0
rocket_x, rocket_y = 0, 0
rocket_vx, rocket_vy = 0, 0
state = "ON_EARTH"

for i in range(num_steps):
    ex, ey = earth_pos[i]
    ax, ay = ast_pos[i]

    # LAUNCH ON THE OPTIMAL DAY
    if i == best_day:
        state = "IN_FLIGHT"
        rocket_x, rocket_y = ex, ey
        
        # Re-calculate the vector for the real simulation
        earth_vx = (earth_pos[i][0] - earth_pos[i-1][0]) / dt
        earth_vy = (earth_pos[i][1] - earth_pos[i-1][1]) / dt
        e_speed = math.sqrt(earth_vx**2 + earth_vy**2)
        
        rocket_vx = earth_vx + (earth_vx / e_speed) * kick_velocity
        rocket_vy = earth_vy + (earth_vy / e_speed) * kick_velocity
        
        print(f"LIFT OFF on Day {i}!")

        current_vx = 0
        current_vy = 0

    # PHYSICS ENGINE
    if state == "ON_EARTH":
        rocket_x, rocket_y = ex, ey
        
    elif state == "IN_FLIGHT":
        r_rocket = math.sqrt(rocket_x**2 + rocket_y**2)
        acc = - (G * mass_of_sun) / r_rocket**3
        
        rocket_vx += acc * rocket_x * dt
        rocket_vy += acc * rocket_y * dt
        rocket_x += rocket_vx * dt
        rocket_y += rocket_vy * dt
        
        # Check for Landing (Increased tolerance to 5 million km for visualization)
        dist_to_ast = math.sqrt((rocket_x - ax)**2 + (rocket_y - ay)**2)
        if dist_to_ast < 5e9: 
            state = "LANDED"
            landing_day = i
            print(f"*** INTERCEPT CONFIRMED on Day {i} ***")

    elif state == "LANDED":
        rocket_x, rocket_y = ax, ay
        
    x_rocket_hist.append(rocket_x)
    y_rocket_hist.append(rocket_y)
    vx_rocket_hist.append(current_vx)
    vy_rocket_hist.append(current_vy)


# 6. ANIMATION 
fig, ax = plt.subplots(figsize=(8,8))
limit = 5.0 * AU 
ax.set_xlim(-limit, limit)
ax.set_ylim(-limit, limit)
ax.set_aspect('equal')
ax.set_facecolor('black')

ax.plot(0, 0, 'yo', markersize=12, label='Sun')

# Lines and Points
line_earth, = ax.plot([], [], 'c-', lw=1, alpha=0.5, label='Earth')
point_earth, = ax.plot([], [], 'co', markersize=5)

line_ast, = ax.plot([], [], 'r-', lw=1, alpha=0.5, label='Asteroid 1986 DA')
point_ast, = ax.plot([], [], 'ro', markersize=4)

line_rocket, = ax.plot([], [], 'm-', lw=1.5, label='Rocket') 
point_rocket, = ax.plot([], [], 'xw', markersize=7) # White 'X' for rocket

user_text = ax.text(0.05, 0.96, '', transform=ax.transAxes, color='white', fontsize=10)
user_duration_of_sim = ax.text(0.05, 0.92, '', transform=ax.transAxes, color='gray', fontsize=10)
day_text = ax.text(0.05, 0.87, '', transform=ax.transAxes, color='blue', fontsize=10)
rel_vel_text = ax.text(0.05, 0.83, '', transform=ax.transAxes, color='green', fontsize=10)



def update(frame):
    # Earth
    line_earth.set_data(x_history[:frame], y_history[:frame])
    point_earth.set_data([x_history[frame]], [y_history[frame]])
    
    # Asteroid
    line_ast.set_data(x_ast_history[:frame], y_ast_history[:frame])
    point_ast.set_data([x_ast_history[frame]], [y_ast_history[frame]])
    
    # Rocket
    line_rocket.set_data(x_rocket_hist[:frame], y_rocket_hist[:frame])
    point_rocket.set_data([x_rocket_hist[frame]], [y_rocket_hist[frame]])

    # User Inputs
    user_text.set_text(f"User: {user_input}")
    user_duration_of_sim.set_text(f"Duration: {user_years} Years")
    # Time Tracker
    day_text.set_text(f"Day: {frame}")
    
    # Relative Rocket Velocity
    if frame > 0:
        dx = x_rocket_hist[frame] - x_rocket_hist[frame-1]
        dy = y_rocket_hist[frame] - y_rocket_hist[frame-1]
        
        # Distance (meters) / Time (seconds)
        speed_mps = math.sqrt(dx**2 + dy**2) / dt
        speed_kps = speed_mps / 1000
    else:
        speed_kps = 0.0
        
    rel_vel_text.set_text(f"Relative Rocket Speed: {speed_kps:.1f} km/s")


    
    return line_earth, point_earth, line_ast, point_ast, line_rocket, point_rocket, user_text, day_text, rel_vel_text

# Skip frames to keep animation fast 
ani = FuncAnimation(fig, update, frames=range(0, num_steps, graphics_quality), interval=120, blit=True)

plt.legend(loc="upper right")
plt.title("Mission to Asteroid 1986 DA")
plt.close()
HTML(ani.to_jshtml())
writer = PillowWriter(fps = 30) 
ani.save("Live_Control_Sim.gif", writer=writer,)

Welcome to Asteroid Mining Simulation version 3.1
Target Asteroid: 6178 (1986 DA)
Requesting Primary Parameters...


Enter Username: Nolan Marsh


Initializing
-------------------------------


Nolan Marsh, enter your desiered simulation duration in years (10-80):  40


Simulation Accuracy is scaled exponentially, please enter in accordance with simulation duration Ex:
If your simulation lasts 40 years, scale your simulation accuracy to around 60%
If your simulation lasts 50 years, scale your simulation accuracy to around 40%
**************** WARNING ****************
 If you set your simulation accuracy too high with a large simulation duration, your system may crash
*****************************************


Nolan Marsh, enter your desiered simulation accuracy in percent accuracy (10-90):  60


Initializing Orbit Calculation
Complete
-------------------------------
 -Mission Control Startup- 
Closest approach: Day 12060
Scanning launch windows from Day 11960 to 12160...
OPTIMAL LAUNCH FOUND: Day 12045
Projected miss distance: 385577 km
-------------------------------
LIFT OFF on Day 12045!
*** INTERCEPT CONFIRMED on Day 12604 ***
